# Reporte Final — DS Airlines

**TPI Ciencia de Datos**

Informe completo: [`docs/TPI_Ciencia_de_Datos.pdf`](../docs/TPI_Ciencia_de_Datos.pdf)

Este notebook ejecuta el pipeline completo del trabajo en un único flujo, partiendo de los tres datasets originales (`Vuelos.xlsx`, `Clientes.xlsx`, `casosNuevos.xlsx`) y produciendo el reporte final solicitado en el enunciado, con una fila por cada par vuelo–pasajero y las columnas:

| ID vuelo | ID pasajero | Predicho_demora_vuelo | Tipo beneficio |

El pipeline replica exactamente las decisiones documentadas en el informe del TPI y el contenido de los notebooks `04_Aplicacion_Arbol_CasosNuevos.ipynb` y `05_KMedias_ClientesDemorados.ipynb`:

1. **Clasificación** con Árbol de Decisión (`criterion='gini'`, `max_depth=5`, `random_state=42`) reentrenado sobre los 15.000 vuelos completos.
2. **Cruce** con `VuelosNuevos_Clientes` para identificar los 30.518 pasajeros únicos afectados por al menos un vuelo demorado.
3. **Segmentación** con K-Medias (`k=3`) sobre las 6 variables numéricas estandarizadas con z-score de esos 30.518 clientes.
4. **Asignación de beneficios**: Cliente Premium → Sala VIP; Cliente Adulto Frecuente y Cliente Joven Ocasional → Voucher comida.
5. **Construcción del reporte final** sobre la totalidad de la tabla relacional `VuelosNuevos_Clientes`, completando con "-" para los vuelos no demorados.

No se generan gráficos: la salida es un único archivo Excel con la tabla final.

## 1. Importaciones y configuración

In [ ]:
import pandas as pd
import numpy as np

from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.cluster import KMeans

RANDOM_STATE = 42

# Rutas a los datasets originales.
# Si se ejecuta en Google Colab, subir los tres archivos antes de correr este notebook.
# from google.colab import files
# files.upload()
RUTA_VUELOS        = 'Vuelos.xlsx'
RUTA_CLIENTES      = 'Clientes.xlsx'
RUTA_CASOS_NUEVOS  = 'casosNuevos.xlsx'
ARCHIVO_SALIDA     = 'reporte_final_DSAirlines.xlsx'

## 2. Funciones del pipeline

Se encapsulan las dos transformaciones clave: la preparación de la vista minable de vuelos (idéntica a la documentada en la Fase 3) y la asignación robusta de nombres a los clusters de K-Medias (no depende del número arbitrario que asigne scikit-learn).

In [ ]:
def hora_a_decimal(h):
    """Convierte una hora en formato 'HH:MM' a número decimal (ej.: '09:30' -> 9.5)."""
    if isinstance(h, str):
        hh, mm = h.split(':')
        return int(hh) + int(mm) / 60
    return h


def preparar_vista_minable_vuelos(df):
    """
    Aplica el preprocesamiento de la Fase 3 sobre el dataset de vuelos.
    Sirve tanto para Vuelos.xlsx (con columna 'demora') como para VuelosNuevos (sin ella).
    """
    df = df.copy()

    cols_excluidas = ['aeropuerto_origen', 'aeropuerto_destino', 'tipo_avion',
                      'puerta_embarque', 'tiempo_estimado_vuelo']
    df = df.drop(columns=cols_excluidas)

    df['hora_salida_programada'] = df['hora_salida_programada'].apply(hora_a_decimal)
    df['temporada_alta'] = df['temporada_alta'].astype(int)

    df = pd.get_dummies(df, columns=['dia_semana', 'condiciones_climaticas',
                                     'congestion_aerea'], drop_first=False, dtype=int)

    referencias = ['dia_semana_Lunes', 'condiciones_climaticas_Despejado',
                   'congestion_aerea_Baja']
    df = df.drop(columns=referencias, errors='ignore')

    return df


def asignar_nombres_clusters(centroides_orig):
    """
    Asigna nombres descriptivos a cada cluster en base a las características de sus
    centroides en escala original. Esta lógica es robusta: si el orden numérico de los
    clusters cambia (otra semilla, otra versión de scikit-learn), los nombres siguen
    correspondiendo al perfil correcto.

    - Cliente Premium       : el cluster con mayor gasto_acumulado promedio.
    - Cliente Joven Ocasional: de los dos restantes, el de menor edad promedio.
    - Cliente Adulto Frecuente: el otro restante (mayor edad promedio).
    """
    nombres = {}
    cluster_premium = int(centroides_orig['gasto_acumulado'].idxmax())
    nombres[cluster_premium] = 'Cliente Premium'

    otros = [c for c in centroides_orig.index if c != cluster_premium]
    edades = centroides_orig.loc[otros, 'edad']
    nombres[int(edades.idxmin())] = 'Cliente Joven Ocasional'
    nombres[int(edades.idxmax())] = 'Cliente Adulto Frecuente'

    return nombres

## 3. Parte 1 — Clasificación de vuelos con el Árbol de Decisión

Se reentrena el modelo final sobre la totalidad de `Vuelos.xlsx` (sin partición 70/30, dado que la evaluación ya fue realizada en la Etapa 1) y se aplica sobre la hoja `VuelosNuevos` de `casosNuevos.xlsx`.

In [ ]:
# Carga de Vuelos.xlsx
vuelos = pd.read_excel(RUTA_VUELOS)
print(f'Dataset de entrenamiento: {vuelos.shape}')

Dataset de entrenamiento: (15000, 15)


In [ ]:
# Aplicar pipeline al dataset de entrenamiento
vuelos_prep = preparar_vista_minable_vuelos(vuelos)

X_train = vuelos_prep.drop(columns=['id_vuelo', 'demora'])
y_train = vuelos_prep['demora']

# Escalado min-max sobre las variables numéricas continuas
cols_numericas_vuelos = ['hora_salida_programada', 'distancia_vuelo',
                         'ocupacion_vuelo', 'visibilidad']

scaler_vuelos = MinMaxScaler()
X_train_scaled = X_train.copy()
X_train_scaled[cols_numericas_vuelos] = scaler_vuelos.fit_transform(
    X_train[cols_numericas_vuelos]
)

# Entrenamiento del árbol final
arbol_final = DecisionTreeClassifier(
    criterion='gini',
    max_depth=5,
    random_state=RANDOM_STATE
)
arbol_final.fit(X_train_scaled, y_train)

print(f'Árbol final reentrenado | Profundidad: {arbol_final.get_depth()} | '
      f'Hojas: {arbol_final.get_n_leaves()}')

Árbol final reentrenado | Profundidad: 5 | Hojas: 32


In [ ]:
# Carga de casosNuevos.xlsx (las dos hojas relevantes)
vuelos_nuevos       = pd.read_excel(RUTA_CASOS_NUEVOS, sheet_name='VuelosNuevos')
rel_vuelo_cliente   = pd.read_excel(RUTA_CASOS_NUEVOS, sheet_name='VuelosNuevos_Clientes')

print(f'VuelosNuevos:           {vuelos_nuevos.shape}')
print(f'VuelosNuevos_Clientes:  {rel_vuelo_cliente.shape}')

VuelosNuevos:           (5000, 14)
VuelosNuevos_Clientes:  (356416, 2)


In [ ]:
# Aplicar pipeline a casos nuevos
nuevos_prep = preparar_vista_minable_vuelos(vuelos_nuevos)
X_nuevos    = nuevos_prep.drop(columns=['id_vuelo'])

# Alineación de columnas con las de entrenamiento (orden y nombres idénticos)
X_nuevos = X_nuevos.reindex(columns=X_train.columns, fill_value=0)

# Escalado con el scaler ya ajustado en entrenamiento (transform, no fit_transform)
X_nuevos_scaled = X_nuevos.copy()
X_nuevos_scaled[cols_numericas_vuelos] = scaler_vuelos.transform(
    X_nuevos[cols_numericas_vuelos]
)

# Predicción
predicciones = arbol_final.predict(X_nuevos_scaled)

# Tabla de vuelos clasificados
vuelos_clasificados = vuelos_nuevos[['id_vuelo']].copy()
vuelos_clasificados['predicho_demora'] = predicciones

print(f'Vuelos clasificados como demorados:    {(predicciones == 1).sum()}')
print(f'Vuelos clasificados como no demorados: {(predicciones == 0).sum()}')

Vuelos clasificados como demorados:    659
Vuelos clasificados como no demorados: 4341


## 4. Parte 2 — Segmentación de clientes demorados con K-Medias

A partir de los vuelos predichos como demorados se obtienen los clientes únicos afectados, y sobre ese subconjunto se entrena K-Medias con `k=3` para asignar a cada uno un perfil que determinará el beneficio compensatorio.

In [ ]:
# Vuelos demorados → clientes únicos afectados
ids_vuelos_demorados = vuelos_clasificados.loc[
    vuelos_clasificados['predicho_demora'] == 1, 'id_vuelo'
]
rel_demorados = rel_vuelo_cliente[
    rel_vuelo_cliente['id_vuelo'].isin(ids_vuelos_demorados)
].copy()
ids_clientes_demorados = rel_demorados['id_cliente'].drop_duplicates()

print(f'Vuelos demorados:                  {len(ids_vuelos_demorados)}')
print(f'Registros vuelo-cliente afectados: {len(rel_demorados)}')
print(f'Clientes únicos a segmentar:       {len(ids_clientes_demorados)}')

Vuelos demorados:                  659
Registros vuelo-cliente afectados: 47247
Clientes únicos a segmentar:       30518


In [ ]:
# Carga de Clientes.xlsx y recuperación de atributos solo de los clientes a segmentar
clientes = pd.read_excel(RUTA_CLIENTES)
clientes_a_analizar = clientes.merge(
    ids_clientes_demorados.to_frame(),
    on='id_cliente',
    how='inner'
)
print(f'Clientes a segmentar: {clientes_a_analizar.shape}')

Clientes a segmentar: (30518, 14)


In [ ]:
# Vista minable para clustering: solo las seis variables numéricas
cols_numericas_clientes = ['edad', 'cant_vuelos', 'cantidad_millas', 'ingreso_mensual',
                           'anticipacion_compra_promedio', 'gasto_acumulado']

X_clientes = clientes_a_analizar[cols_numericas_clientes].copy()

# Estandarización z-score
scaler_clientes = StandardScaler()
X_clientes_scaled = scaler_clientes.fit_transform(X_clientes)

# Entrenamiento del modelo final con k = 3
kmeans_final = KMeans(
    n_clusters=3,
    init='k-means++',
    n_init=10,
    max_iter=300,
    random_state=RANDOM_STATE
)
clusters = kmeans_final.fit_predict(X_clientes_scaled)

# Centroides en escala original y asignación robusta de nombres
centroides_orig = pd.DataFrame(
    scaler_clientes.inverse_transform(kmeans_final.cluster_centers_),
    columns=cols_numericas_clientes
)
nombres_clusters = asignar_nombres_clusters(centroides_orig)

print('Asignación de nombres a clusters:')
for k_, v in sorted(nombres_clusters.items()):
    print(f'  Cluster {k_} → {v}')

# Asignación de segmento a cada cliente
clientes_segmentados = clientes_a_analizar[['id_cliente']].copy()
clientes_segmentados['segmento'] = pd.Series(clusters).map(nombres_clusters).values

print('\nDistribución de clientes por segmento:')
print(clientes_segmentados['segmento'].value_counts().to_string())

Asignación de nombres a clusters:
  Cluster 0 → Cliente Joven Ocasional
  Cluster 1 → Cliente Premium
  Cluster 2 → Cliente Adulto Frecuente

Distribución de clientes por segmento:
segmento
Cliente Adulto Frecuente    14216
Cliente Joven Ocasional     13648
Cliente Premium              2654


## 5. Parte 3 — Asignación de beneficios y construcción del reporte final

Mapeo de segmento → beneficio:

- **Cliente Premium** → Sala VIP
- **Cliente Adulto Frecuente** → Voucher comida
- **Cliente Joven Ocasional** → Voucher comida

La tabla final tiene una fila por cada par vuelo–pasajero de `VuelosNuevos_Clientes`. Para los vuelos no demorados, el `Tipo beneficio` se completa con `"-"` (el cliente puede o no estar segmentado: para los vuelos no demorados nunca le corresponde un beneficio).

In [ ]:
# Mapeo de segmento a beneficio
MAPEO_BENEFICIO = {
    'Cliente Premium':          'Sala VIP',
    'Cliente Adulto Frecuente': 'Voucher comida',
    'Cliente Joven Ocasional':  'Voucher comida',
}

clientes_segmentados['beneficio'] = clientes_segmentados['segmento'].map(MAPEO_BENEFICIO)

# Reporte final: arrancamos desde la tabla relacional completa de casosNuevos
reporte = rel_vuelo_cliente.copy()

# Cruce con la predicción de demora del vuelo
reporte = reporte.merge(vuelos_clasificados, on='id_vuelo', how='left')

# Cruce con el segmento del cliente (solo existe para clientes con vuelos demorados)
reporte = reporte.merge(clientes_segmentados[['id_cliente', 'beneficio']],
                        on='id_cliente', how='left')

# Predicho_demora_vuelo: SI / NO
reporte['Predicho_demora_vuelo'] = np.where(reporte['predicho_demora'] == 1, 'SI', 'NO')

# Tipo beneficio: solo se asigna si el vuelo está predicho como demorado;
# en cualquier otro caso queda en '-'.
reporte['Tipo beneficio'] = np.where(
    reporte['predicho_demora'] == 1,
    reporte['beneficio'],
    '-'
)

# Renombre final a las cabeceras de la imagen y selección de columnas
reporte_final = reporte.rename(columns={
    'id_vuelo':   'ID vuelo',
    'id_cliente': 'ID pasajero',
})[['ID vuelo', 'ID pasajero', 'Predicho_demora_vuelo', 'Tipo beneficio']]

print(f'Reporte final: {reporte_final.shape[0]} filas (una por par vuelo-pasajero)')
reporte_final.head(10)

Reporte final: 356416 filas (una por par vuelo-pasajero)


,ID vuelo,ID pasajero,Predicho_demora_vuelo,Tipo beneficio
0,20651,25273,NO,-
1,20651,22816,NO,-
2,20651,27405,NO,-
3,20651,33494,NO,-
4,20651,26181,NO,-
5,20651,29645,NO,-
6,20651,30687,NO,-
7,20651,32752,NO,-
8,20651,34127,NO,-
9,20651,29186,NO,-


In [ ]:
# Verificaciones rápidas de coherencia
print('Distribución de Predicho_demora_vuelo en el reporte:')
print(reporte_final['Predicho_demora_vuelo'].value_counts().to_string())

print('\nDistribución de Tipo beneficio en el reporte:')
print(reporte_final['Tipo beneficio'].value_counts().to_string())

print('\nDistribución de Tipo beneficio solo entre vuelos demorados:')
print(
    reporte_final.loc[reporte_final['Predicho_demora_vuelo'] == 'SI', 'Tipo beneficio']
    .value_counts().to_string()
)

Distribución de Predicho_demora_vuelo en el reporte:
Predicho_demora_vuelo
NO    309169
SI     47247

Distribución de Tipo beneficio en el reporte:
Tipo beneficio
-                 309169
Voucher comida     43203
Sala VIP            4044

Distribución de Tipo beneficio solo entre vuelos demorados:
Tipo beneficio
Voucher comida    43203
Sala VIP           4044


## 6. Exportación

In [ ]:
with pd.ExcelWriter(ARCHIVO_SALIDA, engine='openpyxl') as writer:
    reporte_final.to_excel(writer, sheet_name='asignacion_beneficios', index=False)

    # Hoja adicional: copia de la principal filtrada a vuelos predichos como demorados
    reporte_demorados = reporte_final[reporte_final['Predicho_demora_vuelo'] == 'SI']
    reporte_demorados.to_excel(writer, sheet_name='solo_demorados', index=False)

print(f'Archivo generado: {ARCHIVO_SALIDA}')
print(f'  - asignacion_beneficios: {len(reporte_final)} filas')
print(f'  - solo_demorados:        {len(reporte_demorados)} filas')

Archivo generado: reporte_final_DSAirlines.xlsx
  - asignacion_beneficios: 356416 filas
  - solo_demorados:        47247 filas
